# MAFIA Training on Google Colab

This notebook allows you to train MAFIA Observer + TD3 on Google Colab with GPU acceleration.

**Requirements:**
- Upload your code to GitHub or Google Drive
- Upload `stock_data.csv` to Google Drive

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

# Check RAM
!free -h

## 2. Mount Google Drive (if using Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Code (Option A: GitHub)

In [ ]:
# Replace with your GitHub repo URL
!git clone https://github.com/YOUR_USERNAME/the_last_dance.git
%cd the_last_dance/agents/MAFIA

## 3. Copy Code (Option B: Google Drive)

In [ ]:
# If you uploaded zip to Drive
!cp /content/drive/MyDrive/the_last_dance.zip .
!unzip -q the_last_dance.zip
%cd the_last_dance/agents/MAFIA

## 4. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q stable-baselines3
!pip install -q pandas numpy matplotlib seaborn
!pip install -q tensorboard

# Optional: TA-Lib (for technical indicators)
!wget http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
!tar -xzf ta-lib-0.4.0-src.tar.gz
%cd ta-lib
!./configure --prefix=/usr
!make
!make install
%cd ..
!pip install -q TA-Lib

## 5. Copy Data Files

In [ ]:
# Option 1: From Drive (if you uploaded CSV files)
!cp /content/drive/MyDrive/stock_data_top23.csv data/
!cp /content/drive/MyDrive/stock_data_dynamic143.csv data/

# Create symlink to use top23 by default (for 12GB Colab RAM)
!ln -sf stock_data_top23.csv data/stock_data.csv

# Check data
!ls -lh data/

## 6. Run Training - Phase 1 (Observer)

In [ ]:
# Train Observer with small scope (2020-2022)
!python scripts/train_separated.py --phase 1 \
    --start-year 2020 \
    --first-infer-year 2022 \
    --last-infer-year 2022 \
    --output-dir ./colab_results \
    --no-display

## 7. Run Training - Full Pipeline (2015-2022)

In [ ]:
# Full training pipeline (if Colab RAM is enough)
# Use dynamic143 if you have Colab Pro (25GB RAM)
# !ln -sf stock_data_dynamic143.csv data/stock_data.csv

!python scripts/train_separated.py --phase all \
    --start-year 2015 \
    --first-infer-year 2018 \
    --last-infer-year 2022 \
    --output-dir ./colab_results \
    --no-display

## 8. Monitor Training (TensorBoard)

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Start TensorBoard
%tensorboard --logdir ./colab_results/tb_logs

## 9. Download Results

In [ ]:
# Zip results
!zip -r colab_results.zip ./colab_results

# Copy to Drive
!cp colab_results.zip /content/drive/MyDrive/

# Or download directly
from google.colab import files
files.download('colab_results.zip')

## 10. Check Memory Usage

In [ ]:
# Monitor RAM
!free -h

# Monitor GPU
!nvidia-smi

# Python memory profiling
import psutil
import os

process = psutil.Process(os.getpid())
print(f"RAM usage: {process.memory_info().rss / 1024**3:.2f} GB")